In [1]:
import os
import sys

# Tell Jupyter to use Git Bash's shell engine
os.environ['SHELL'] = r'C:\Program Files\Git\bin\sh.exe'

# Set HADOOP_HOME to the parent folder of 'bin'
os.environ['HADOOP_HOME'] = "C:/hadoop"
# Add the bin folder to the system PATH
os.environ['PATH'] += os.pathsep + "C:/hadoop/bin"

# Get the path to the current python executable in your .venv
python_path = sys.executable

os.environ['PYSPARK_PYTHON'] = python_path
os.environ['PYSPARK_DRIVER_PYTHON'] = python_path

In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

# Question 1: Install Spark and PySpark
Install Spark
Run PySpark
Create a local spark session
Execute spark.version.
What's the output?

In [38]:
spark.version

'4.1.1'

# Question 2: Yellow November 2025
Read the November 2025 Yellow into a Spark Dataframe.

Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

In [ ]:
import wget

url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet'
filename = wget.download(url)

In [4]:
df = spark.read \
    .option("header", "true") \
    .parquet('yellow_tripdata_2025-11.parquet')

In [5]:
df = df.repartition(4)

In [6]:
df.write.mode("overwrite").parquet('yellow/2025/11/')

In [7]:
df_yellow = spark.read.parquet('yellow/2025/11/')

# Question 3: Count records
How many taxi trips were there on the 15th of November?

Consider only trips that started on the 15th of November.

In [8]:
df_yellow = df_yellow \
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')

In [9]:
df_yellow.createOrReplaceTempView('yellow_trips_data')

In [11]:
df_yellow \
    .withColumn('pickup_date', F.to_date(df_yellow.pickup_datetime)) \
    .filter("pickup_date = '2025-11-15'") \
    .count()

162604

# Question 4: Longest trip
What is the length of the longest trip in the dataset in hours?

In [ ]:
df_yellow.withColumn("duration_hours", 
    (F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")) / 3600
).select(F.max("duration_hours")).show()

+-------------------+
|max(duration_hours)|
+-------------------+
|  90.64666666666666|
+-------------------+



# Question 5: User Interface
Spark's User Interface which shows the application's dashboard runs on which local port?

4040

# Question 6: Least frequent pickup location zone
Load the zone lookup data into a temp view in Spark:

wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

In [ ]:
url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'
filename = wget.download(url)

In [55]:
zones = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

In [60]:
df_yellow.columns

['VendorID',
 'pickup_datetime',
 'dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [58]:
zones.columns

['LocationID', 'Borough', 'Zone', 'service_zone']

In [57]:
zones.createOrReplaceTempView('zones')

In [61]:
spark.sql("""
SELECT
   Zone, COUNT(*) as total_zone
FROM yellow_trips_data t
JOIN zones z on t.PULocationID = z.LocationID
GROUP BY 
    Zone
ORDER BY
    total_zone;
""").show()

+--------------------+----------+
|                Zone|total_zone|
+--------------------+----------+
|Eltingville/Annad...|         1|
|Governor's Island...|         1|
|       Arden Heights|         1|
|       Port Richmond|         3|
|       Rikers Island|         4|
|   Rossville/Woodrow|         4|
|         Great Kills|         4|
| Green-Wood Cemetery|         4|
|         Jamaica Bay|         5|
|         Westerleigh|        12|
|New Dorp/Midland ...|        14|
|       West Brighton|        14|
|             Oakwood|        14|
|        Crotona Park|        14|
|       Willets Point|        15|
|Breezy Point/Fort...|        16|
|Saint George/New ...|        17|
|       Broad Channel|        18|
|     Mariners Harbor|        21|
|Heartland Village...|        22|
+--------------------+----------+
only showing top 20 rows
